# Event Data Generation Notebook

Turn match **tracking data** into match **event data** (passes, shots, goals, set pieces, challenges...).

This notebook implements the methodology from [*Generating Footballing Event Data from Match Tracking Data*](https://medium.com/@johncomonitski/generating-footballing-event-data-from-match-tracking-data-1730c58ad598):
1. **Possession tracking** &mdash; a possession begins when the ball is within the possession radius of its closest player, and ends once the ball leaves that player. Noise from momentary ball movement and failed tackles is filtered out.
2. **Event detection** &mdash; the possession list is walked with a sliding `(previous, current, next)` window. `review_events` inspects each triple and emits the appropriate event(s).

**Imports**

The event generator lives alongside the tracking `Match` library in `data_cleanup/lib`, so we add it to the path.

In [ ]:
import sys
sys.path.insert(0, "./../data_cleanup")
from lib.match import Match

**Tracking Data Import**

Point the notebook at the tracking data you want to generate events from. This can be raw tracking output or cleaned tracking data from the [Tracking Data Clean Up Notebook](./../data_cleanup/cleanup.ipynb).

Use `import_metrica` for Metrica-format (normalised) data or `import_raw_data` for the pipeline's raw pitch-coordinate CSVs.

In [ ]:
PATH = "./../data_cleanup/output/"
FILE_NAME = ""

match = Match()
match.import_metrica(PATH, FILE_NAME)
# For raw tracking pipeline output instead, use:
# match.import_raw_data(PATH, FILE_NAME)

**Generate Events**

`match.generate_events()` returns an `EventLog`. The possession radius (metres on a standard 105 x 68 m pitch) can be tuned for your data &mdash; lower it for cleaner tracking, raise it for noisier tracking.

In [ ]:
events = match.generate_events()

events.print()
print("\nSummary:", events.summary())

**Inspect Specific Events**

Filter the log by event type to drill into what was detected.

In [ ]:
for shot in events.filter("SHOT"):
    print(shot)

# The detected possessions are available too:
# for p in match.event_generator.possessions:
#     print(p)

**Export Event Data**

Exports to the Metrica event-data CSV format in `event_generation/output/`.

In [ ]:
events.export(path="./output/", file_name=(FILE_NAME or "events").rsplit(".", 1)[0] + "_events.csv")